In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

mlflow.set_registry_uri("databricks-uc")

In [0]:
train_df = spark.table("ecommerce.ml.customer_training_base").toPandas()

display(spark.table("ecommerce.ml.customer_training_base").limit(10))
print(train_df.shape)
print(train_df["label_buy_next_30d"].value_counts(dropna=False))

In [0]:
feature_cols = [
    "orders_hist",
    "items_hist",
    "revenue_hist_inr",
    "avg_order_value_hist_inr",
    "avg_discount_percent_hist",
    "coupon_orders_hist",
    "distinct_channels_hist",
    "recency_days"
]

target_col = "label_buy_next_30d"

df = train_df.dropna(subset=feature_cols + [target_col]).copy()

if df[target_col].nunique() < 2:
    raise ValueError("Training data has only one class. You need both 0 and 1 labels to train this model.")

In [0]:
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

experiment_path = "/Shared/ecommerce_purchase_propensity"
mlflow.set_experiment(experiment_path)

with mlflow.start_run():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_param("feature_cols", ",".join(feature_cols))
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("roc_auc", auc)
    input_example = X_train.head(5)
    proba_example = model.predict_proba(input_example)
    signature = infer_signature(input_example, model.predict(input_example))

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="ecommerce.ml.purchase_propensity_model",
        signature=signature,
        input_example=input_example,
        pyfunc_predict_fn="predict_proba"
    )

    print("accuracy:", acc)
    print("roc_auc:", auc)